# SVD and PCA

## What's covered

- **Singular Value Decomposition (SVD)** — `A = U Σ V^T`, works for *every* matrix
- **Geometric interpretation** — rotate, scale, rotate
- The link between SVD and eigendecomposition of `A^T A` and `A A^T`
- **Singular values vs eigenvalues** — when they agree, when they differ
- **Low-rank approximation** and the **Eckart-Young theorem** — SVD is the optimal compressor
- **PCA**, derived cleanly from SVD on centered data
- The **pseudoinverse** `A^+` — the universal solver via SVD
- Where this appears in ML — recommender systems, embeddings, transformers (LoRA), image compression


## The Singular Value Decomposition

Eigendecomposition is beautiful but restrictive: only square, only diagonalizable. **SVD lifts both restrictions.** Every matrix — square or rectangular, full rank or rank-deficient, real or complex — has a singular value decomposition:

$$
A = U \Sigma V^T
$$

For an `m × n` matrix `A`:

- `U` is `m × m`, **orthogonal**. Its columns are the **left singular vectors** `u_i`.
- `Σ` is `m × n`, **diagonal** (with zeros padding the non-square part). Diagonal entries `σ_1 ≥ σ_2 ≥ ... ≥ 0` are the **singular values**.
- `V` is `n × n`, **orthogonal**. Its columns are the **right singular vectors** `v_i`.

The number of non-zero singular values equals the **rank** of `A`. Trailing zeros expose rank deficiency directly.

**Geometric picture.** `A` takes a unit sphere in `R^n` to an *ellipsoid* in `R^m`. The SVD reveals exactly which axes and stretch factors define that ellipsoid:

1. `V^T` rotates the sphere to align it with the right singular vectors.
2. `Σ` stretches each axis by `σ_i`.
3. `U` rotates the resulting ellipsoid into its final position in `R^m`.

Every linear transformation is one rotation, one scaling, one rotation. That's SVD.


In [ ]:
import numpy as np

# A rectangular matrix — eigendecomposition cannot handle this, but SVD can
A = np.array([[3.0, 1.0, 1.0],
              [-1.0, 3.0, 1.0]])

U, s, Vt = np.linalg.svd(A, full_matrices=False)   # 'thin' SVD — most useful in practice
print("U shape  =", U.shape)
print("s (singular values) =", s)
print("V^T shape =", Vt.shape)

# Reconstruct A
reconstructed = U @ np.diag(s) @ Vt
print("\nU Σ V^T =\n", reconstructed)
print("matches A?", np.allclose(reconstructed, A))

# Sanity check: U and V have orthonormal columns
print("\nU^T U ≈ I?", np.allclose(U.T @ U, np.eye(2)))
print("V^T V ≈ I?", np.allclose(Vt @ Vt.T, np.eye(2)))


## The link to eigendecomposition

SVD is not random — it is exactly the eigendecomposition of two symmetric matrices in disguise.

**Square the matrix two ways:**

$$
A^T A = (U \Sigma V^T)^T (U \Sigma V^T) = V \Sigma^T U^T U \Sigma V^T = V \Sigma^T \Sigma V^T = V \, \Sigma^2 \, V^T
$$

So `A^T A` is symmetric, and its eigendecomposition has:

- **eigenvectors** = right singular vectors of `A` (columns of `V`)
- **eigenvalues** = squared singular values `σ_i^2`

Similarly, `A A^T = U Σ² U^T` has:

- **eigenvectors** = left singular vectors of `A` (columns of `U`)
- **eigenvalues** = squared singular values `σ_i^2`

This is the bridge between Notebook 7 (eigendecomposition) and SVD. **For a symmetric positive semidefinite matrix, SVD coincides with eigendecomposition** (and singular values equal eigenvalues exactly, no squaring). For a general matrix, you can think of SVD as the eigendecomposition of `A^T A` (or `A A^T`) — but worked out without ever forming the product, which is numerically smarter.

**Practical consequence.** Whenever you see "eigenvalues of the covariance matrix" in ML (e.g. PCA), you can swap in "squared singular values of the centered data matrix" — and skip computing the covariance entirely.


In [ ]:
# Verify: singular values of A = sqrt of eigenvalues of A^T A
A = np.array([[3.0, 1.0, 1.0],
              [-1.0, 3.0, 1.0]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)

ATA = A.T @ A
eigvals_ATA = np.linalg.eigvalsh(ATA)       # eigh: symmetric → real, sorted ascending
eigvals_ATA = np.sort(eigvals_ATA)[::-1]    # match SVD's descending order

print("singular values     =", s)
print("sqrt eigenvalues of A^T A =", np.sqrt(np.abs(eigvals_ATA[:len(s)])))
print("match?", np.allclose(s, np.sqrt(np.abs(eigvals_ATA[:len(s)]))))


## Low-rank approximation and Eckart-Young

Here is the result that justifies half of modern ML compression. Write the SVD as a sum:

$$
A = \sum_{i=1}^{r} \sigma_i \, \mathbf{u}_i \mathbf{v}_i^T
$$

Each `u_i v_i^T` is a **rank-1 matrix** (outer product). The full sum recovers `A`. Truncating at `k` terms gives a rank-`k` matrix:

$$
A_k = \sum_{i=1}^{k} \sigma_i \, \mathbf{u}_i \mathbf{v}_i^T
$$

**The Eckart-Young theorem.** `A_k` is the *best* rank-`k` approximation to `A` in both the Frobenius and spectral norms. No other rank-`k` matrix gets closer to `A`. The error is exactly the discarded tail:

$$
\|A - A_k\|_F^2 = \sum_{i=k+1}^{r} \sigma_i^2
$$

This is *the* result behind:

- **Image compression** — keep the top `k` singular values, get a recognizable image at a fraction of the storage.
- **Recommender systems** — user-item rating matrices are approximately low-rank; SVD or matrix-factorization recovers latent factors.
- **LoRA / adapters in LLM fine-tuning** — instead of fully updating a weight matrix `W`, train a low-rank update `W + AB` where `A` and `B` are skinny. Storage drops 100×.
- **Dimensionality reduction** — PCA is exactly this trick applied to centered data.

The key intuition: real data is *almost never* full rank. The singular values usually decay quickly, so a small `k` captures most of the information.


In [ ]:
# Low-rank approximation on a small "image"-like matrix
rng = np.random.default_rng(0)

# Build a matrix that is genuinely rank-2 + noise
left = rng.normal(size=(20, 2))
right = rng.normal(size=(2, 30))
A = left @ right + 0.05 * rng.normal(size=(20, 30))

U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("first 6 singular values:", s[:6].round(3))
print("rank by tolerance       :", np.sum(s > 0.1), " <- the noise singular values are much smaller")

# Reconstruct with k = 1, 2, 5 terms
for k in [1, 2, 5]:
    A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    err = np.linalg.norm(A - A_k, ord='fro')
    eckart_young = np.sqrt(np.sum(s[k:] ** 2))
    print(f"  k={k}: ||A - A_k||_F = {err:.4f},  predicted by Eckart-Young = {eckart_young:.4f}")


## PCA — derived from SVD

**Principal Component Analysis** is the workhorse of dimensionality reduction. We can derive it in three honest lines using SVD.

**Setup.** Let `X` be an `n × d` data matrix (rows = samples, columns = features). **Center** it by subtracting the column means: `X_c = X - mean(X)`. The covariance matrix is

$$
C = \frac{1}{n-1} X_c^T X_c \qquad (\text{symmetric, PSD, } d \times d)
$$

**Goal.** Find directions in feature space that capture the most variance — the **principal components**.

**Solution.** Take the SVD of the centered data:

$$
X_c = U \Sigma V^T
$$

Then:

- The **principal components** are the columns of `V` — they are the right singular vectors of `X_c`, equivalently the eigenvectors of `C`.
- The **variance** along the `i`-th principal component is `σ_i^2 / (n-1)`.
- The **projection of the data onto the top `k` components** is `X_c V_k`, a smaller `n × k` matrix.

Two beats to remember:

1. **Always center first.** Without centering, the first "principal component" picks up the mean direction instead of the variance direction.
2. **Use SVD, not the covariance matrix directly.** Forming `X_c^T X_c` is numerically wasteful and squares the condition number. `np.linalg.svd(X_c)` is the right call.


In [ ]:
# PCA from scratch via SVD
rng = np.random.default_rng(1)

# Synthetic 2D data that's elongated along a 45-degree direction
n = 200
true_axis = np.array([1.0, 1.0]) / np.sqrt(2)   # the direction PCA should recover
X = (rng.normal(scale=3.0, size=n)[:, None] * true_axis +
     0.3 * rng.normal(size=(n, 2)))

# Step 1: center
X_c = X - X.mean(axis=0)

# Step 2: SVD
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)

# Step 3: principal components are rows of V^T (== columns of V)
print("principal components (rows = directions):\n", Vt.round(3))
print("\ntrue elongation axis: ", true_axis.round(3))
print("\nvariance per component:", (s**2 / (n - 1)).round(3))
print("explained variance ratio:", (s**2 / (s**2).sum()).round(3))

# Project onto the top 1 PC -> 1D coordinates
X_1d = X_c @ Vt[0]
print("\nshape after dim. reduction:", X_1d.shape, " <- down from", X.shape)


## The pseudoinverse — the universal solver

For square invertible matrices, `A^{-1}` exists and `x = A^{-1} b` solves `Ax = b`. For everything else — rectangular, rank-deficient, underdetermined — the right generalization is the **Moore-Penrose pseudoinverse** `A^+`, built from SVD:

$$
A = U \Sigma V^T \quad \Longrightarrow \quad A^+ = V \Sigma^+ U^T
$$

where `Σ^+` flips the non-zero singular values (`σ_i ↦ 1/σ_i`) and leaves the zeros alone, then transposes.

**What `A^+` does for you in one move:**

- **Overdetermined system** (more rows than columns, no exact solution): `x = A^+ b` is the least-squares solution.
- **Underdetermined system** (more columns than rows, infinite solutions): `x = A^+ b` is the minimum-norm solution.
- **Square invertible**: `A^+ = A^{-1}` — no surprises.

This single object replaces three different methods from Notebook 5. `np.linalg.lstsq` and `np.linalg.pinv` both use this SVD-based pseudoinverse under the hood.

**Numerical tip.** Tiny singular values get inverted into huge ones, which amplifies noise. In practice, `pinv` zeroes out singular values below a tolerance (`σ < tol · σ_max`). This is **truncated SVD regularization** — closely related to ridge regression in effect.


In [ ]:
# Three cases, one tool
rng = np.random.default_rng(7)

# Overdetermined: 30 equations, 3 unknowns
A_over = rng.normal(size=(30, 3))
b_over = rng.normal(size=30)
x_over = np.linalg.pinv(A_over) @ b_over
x_lstsq, *_ = np.linalg.lstsq(A_over, b_over, rcond=None)
print("overdetermined match (pinv vs lstsq)? ", np.allclose(x_over, x_lstsq))

# Underdetermined: 2 equations, 5 unknowns
A_under = rng.normal(size=(2, 5))
b_under = rng.normal(size=2)
x_under = np.linalg.pinv(A_under) @ b_under
print("underdetermined: x_pinv satisfies A x = b?", np.allclose(A_under @ x_under, b_under))
print("                 ||x_pinv|| =", np.linalg.norm(x_under), "  (minimum-norm solution)")

# Rank-deficient: pinv still works
A_rank1 = np.outer(rng.normal(size=4), rng.normal(size=3))
print("\nrank-deficient matrix rank =", np.linalg.matrix_rank(A_rank1))
print("A^+ shape =", np.linalg.pinv(A_rank1).shape, "  <- handles it without complaint")


## Where this appears in ML

SVD and PCA are everywhere. Once the eight pieces (`A = U Σ V^T`, low-rank truncation, pseudoinverse, PCA) click, you understand the linear core of modern ML.

- **Dimensionality reduction (PCA).** Compress high-dimensional features down to the top `k` principal components. Standard preprocessing for visualization, denoising, and any model that suffers from the curse of dimensionality.
- **Recommender systems.** User-item rating matrices are approximately low-rank — users have a small number of latent preferences. Truncated SVD or matrix factorization (essentially SVD with missing values) recovers these latent factors.
- **LoRA (Low-Rank Adaptation) for LLM fine-tuning.** Replace a full weight update `ΔW` (`d × d`) with `AB` where `A` is `d × r`, `B` is `r × d`, and `r << d`. The Eckart-Young intuition says the most useful directions of `ΔW` are captured by the top singular components — and `r` of them is usually enough.
- **Word2Vec / GloVe.** Truncated SVD on the word co-occurrence matrix is one of the cleanest ways to derive word embeddings — predates the neural-network versions.
- **Image compression / denoising.** Keep top `k` singular values of an image matrix.
- **Pseudoinverse in linear regression.** Universal solver for over- or underdetermined systems. Equivalent to ridge in the limit `λ → 0` after truncating tiny singular values.
- **Spectral norm and Lipschitz bounds.** `||W||_2 = σ_max(W)`. Spectral normalization in GANs literally divides each layer's weight by its largest singular value.
- **Truncated SVD in attention.** Some efficient transformer variants (Linformer, Nyströmformer) approximate attention matrices via low-rank decompositions — direct applications of Eckart-Young.
- **Numerical rank estimation.** Singular values reveal effective rank even for noisy matrices; eigenvalues do not (and `np.linalg.matrix_rank` is built on SVD).

That closes the linear algebra repo. From a `2x + y = 5` warm-up to LoRA in eight notebooks. Next on the math foundations roadmap: **calculus** (gradients, Hessians, optimization) and **probability & statistics** (distributions, MLE, expectation). Then the **machine learning** repo builds on all three.
